# Valorant Skin Bundle — Reddit Sentiment Analysis
**Purchase Intent Detection using NLP & VADER Sentiment Analysis**

**Author:** Chetan Prakash

---

## Project Overview

Valorant is Riot Games' free-to-play tactical shooter. The game generates revenue entirely
through cosmetic skin bundles — priced between $35–$100+ per bundle. Player purchase intent
is a direct driver of revenue.

This project scrapes Reddit comments from Valorant skin bundle announcement posts,
applies NLP preprocessing, and uses VADER sentiment analysis to classify each comment
into one of three purchase intent categories:

- **Will Buy** — positive sentiment, likely to purchase
- **Will Not Buy** — negative sentiment, unlikely to purchase
- **Neutral / Undecided** — no strong signal

## Business Value

Game studios and publishers can use this kind of pipeline to:
- Gauge community reception of a skin bundle **before** finalising pricing
- Track sentiment shifts across multiple bundle releases
- Identify what design or pricing elements drive negative purchase intent
- Build early-warning signals for poorly-received content

## Methodology: Baseline → Improved

This notebook documents two approaches side by side:

**V1 — Baseline (NLP_Project.ipynb, NLP2.ipynb):**
- Counted words from VADER's positive/negative word lists
- Classified as Will Buy (count > 0), Will Not Buy (count < 0), or Wants Changes (count == 0)
- Problems: ignored intensity modifiers, mishandled negations ("not good" = 1 positive word)

**V2 — Improved (this notebook):**
- Better text cleaning (URLs, Reddit quote markers, broader emoji range)
- Uses VADER's compound score directly — handles negations, boosters, ALL-CAPS
- VADER's recommended thresholds (±0.05) instead of counting exactly to zero

## Data

Three Valorant subreddit skin announcement posts were analysed:

| Post | Reddit Submission ID | Comments | Approach |
|------|---------------------|----------|----------|
| Bundle Announcement 1 | `1b4p0vr` | ~381 | V1 Baseline |
| Bundle Announcement 2 | `xibjap`  | ~338 | V1 Baseline |
| Bundle Announcement 3 | `i2ykah`  | 471  | V2 Improved |

---

> **Setup:** Add your Reddit API credentials to a `.env` file or replace
> `YOUR_CLIENT_ID` and `YOUR_CLIENT_SECRET` with your own values.
> Get free API credentials at [reddit.com/prefs/apps](https://www.reddit.com/prefs/apps)

In [ ]:
# Install required library if not already installed
# pip install praw python-dotenv

## 1. Setup & Imports

In [ ]:
import praw
import pandas as pd
import re
import nltk
import os
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data (runs once)
for pkg in ['punkt','punkt_tab','stopwords','wordnet','omw-1.4','vader_lexicon']:
    nltk.download(pkg, quiet=True)

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
print('Libraries loaded successfully')

## 2. Reddit API Connection

> Replace `YOUR_CLIENT_ID` and `YOUR_CLIENT_SECRET` with your Reddit API credentials.
> Never commit real credentials to GitHub.

In [ ]:
# ── REDDIT API CREDENTIALS ────────────────────────────────────────────────
# Option A: hardcode for local use (DO NOT push to GitHub with real values)
CLIENT_ID     = 'YOUR_CLIENT_ID'
CLIENT_SECRET = 'YOUR_CLIENT_SECRET'
USER_AGENT    = 'valorant-sentiment-analysis'

# Option B: load from environment variable (recommended for GitHub)
# CLIENT_ID     = os.environ.get('REDDIT_CLIENT_ID')
# CLIENT_SECRET = os.environ.get('REDDIT_CLIENT_SECRET')

reddit = praw.Reddit(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    user_agent=USER_AGENT
)

print('Reddit connection established')

## 3. Data Collection

In [ ]:
def collect_comments(submission_id):
    """
    Collect all top-level comments from a Reddit submission.

    Parameters
    ----------
    submission_id : str — Reddit post ID (from the URL)

    Returns
    -------
    pd.DataFrame with columns: author, comment
    """
    submission = reddit.submission(id=submission_id)
    submission.comments.replace_more(limit=0)  # expand all comments

    rows = []
    for comment in submission.comments.list():
        rows.append({
            'author':  comment.author.name if comment.author else 'Deleted',
            'comment': comment.body
        })

    df = pd.DataFrame(rows)
    print(f'  Collected {len(df)} comments from post {submission_id}')
    return df


# ── Collect from all three posts ──────────────────────────────────────────
print('Collecting comments...')
df1 = collect_comments('1b4p0vr')   # Bundle Announcement 1
df2 = collect_comments('xibjap')    # Bundle Announcement 2
df3 = collect_comments('i2ykah')    # Bundle Announcement 3

# Label which post each comment came from
df1['post'] = 'Bundle 1'
df2['post'] = 'Bundle 2'
df3['post'] = 'Bundle 3'

# Combined dataframe
all_df = pd.concat([df1, df2, df3], ignore_index=True)
print(f'\nTotal comments collected: {len(all_df)}')
all_df.head(3)

## 4. Text Preprocessing

### V1 vs V2 Comparison — Text Cleaning

**V1 issue:** Only removed emoticons in the range `\U0001F600–\U0001F64F`.
Missed URLs (`https://...`), Reddit quote markers (`> quoted text`),
other symbol ranges, and extra whitespace.

**V2 fix:** Extended emoji range + strips URLs and Reddit quote markers.

In [ ]:
def clean_text(text):
    """
    Clean raw Reddit comment text.
    Removes URLs, Reddit quote markers, emojis, and extra whitespace.
    """
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove Reddit-style quoted lines (lines starting with >)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)
    # Remove emojis — broader range than V1
    emoji_pattern = re.compile(
        '['
        u'\U0001F600-\U0001F64F'   # emoticons
        u'\U0001F300-\U0001F5FF'   # symbols & pictographs
        u'\U0001F680-\U0001F6FF'   # transport & map symbols
        u'\U0001F1E0-\U0001F1FF'   # flag symbols
        ']+', flags=re.UNICODE
    )
    text = emoji_pattern.sub('', text)
    # Collapse extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def tokenize_and_lemmatize(comment):
    """Tokenize, remove stop words, and lemmatize a comment."""
    cleaned = clean_text(comment)
    tokens  = word_tokenize(cleaned)
    filtered = [w for w in tokens if w.lower() not in stop_words and w.isalpha()]
    return [lemmatizer.lemmatize(w) for w in filtered]


all_df['tokens']           = all_df['comment'].apply(tokenize_and_lemmatize)
print('Tokenisation and lemmatisation complete')
print(f'Example tokens: {all_df["tokens"].iloc[0]}')

## 5. Sentiment Analysis

### V1 vs V2 Comparison — Scoring Method

**V1 approach:** Extracted VADER's positive and negative word lists, then counted
how many lemmatized tokens from each comment appeared in each list.
Sentiment score = positive count − negative count.

**Problems with V1:**
- `"not good"` → tokenized as `["not", "good"]` → "good" counted as +1 positive
- `"very bad"` scored the same as `"bad"` — intensity ignored
- `score == 0 exactly` → "Wants Changes" (almost nothing hit exactly 0)

**V2 fix:** Pass the cleaned raw text directly to VADER's `polarity_scores()`.
VADER handles negations, boosters ("very", "extremely"), and ALL-CAPS internally.
Use VADER's recommended thresholds (±0.05) for the neutral zone.

In [ ]:
sid = SentimentIntensityAnalyzer()

# ── V1 APPROACH (for comparison) ─────────────────────────────────────────
ndct = [word for word, score in sid.lexicon.items() if score < 0]  # negative words
pdct = [word for word, score in sid.lexicon.items() if score > 0]  # positive words

def v1_sentiment_score(lemmatized_tokens):
    """V1: raw word count approach."""
    pos = sum(1 for w in lemmatized_tokens if w.lower() in pdct)
    neg = sum(1 for w in lemmatized_tokens if w.lower() in ndct)
    return pos - neg

def v1_label(score):
    """V1: score == 0 → Wants Changes (problematic — few comments hit exactly 0)."""
    if score > 0:  return 'Will Buy'
    elif score < 0: return 'Will Not Buy'
    else:           return 'Wants Changes'


# ── V2 APPROACH (improved) ────────────────────────────────────────────────
POSITIVE_THRESHOLD =  0.05   # VADER's own recommended threshold
NEGATIVE_THRESHOLD = -0.05

def v2_compound_score(comment):
    """V2: VADER compound score on cleaned text."""
    return sid.polarity_scores(clean_text(comment))['compound']

def v2_label(compound):
    """V2: threshold-based — neutral zone captures genuinely ambiguous comments."""
    if compound >= POSITIVE_THRESHOLD:  return 'Will Buy'
    elif compound <= NEGATIVE_THRESHOLD: return 'Will Not Buy'
    else:                                return 'Neutral / Undecided'


# Apply both versions
all_df['V1 Score']         = all_df['tokens'].apply(v1_sentiment_score)
all_df['V1 Label']         = all_df['V1 Score'].apply(v1_label)
all_df['Compound Score']   = all_df['comment'].apply(v2_compound_score)
all_df['Sentiment Label']  = all_df['Compound Score'].apply(v2_label)

print('Sentiment scoring complete')
print(f'Compound score range: {all_df["Compound Score"].min():.3f} to {all_df["Compound Score"].max():.3f}')
print(f'Mean compound score : {all_df["Compound Score"].mean():.4f}')

## 6. Results

In [ ]:
print('=== V2 RESULTS (Improved — All Posts Combined) ===')
v2_overall = all_df['Sentiment Label'].value_counts(normalize=True) * 100
print(v2_overall.to_string())

print('\n=== V2 RESULTS — Per Post ===')
per_post = all_df.groupby('post')['Sentiment Label'].value_counts(
    normalize=True).mul(100).round(1).rename('Percentage')
print(per_post.to_string())

print('\n=== V1 vs V2 LABEL DISTRIBUTION COMPARISON (All Posts) ===')
v1_overall = all_df['V1 Label'].value_counts(normalize=True) * 100
comparison = pd.DataFrame({'V1 (Baseline)': v1_overall, 'V2 (Improved)': v2_overall})
print(comparison.round(1).to_string())

## 7. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Overall V2 sentiment distribution
ax = axes[0]
colors = {'Will Buy':'steelblue', 'Will Not Buy':'tomato', 'Neutral / Undecided':'lightgray'}
v2_overall.plot(kind='bar', ax=ax,
                color=[colors.get(l,'gray') for l in v2_overall.index],
                edgecolor='white', width=0.6)
ax.set_title('Purchase Intent — All Posts\n(V2 Improved)', fontweight='bold')
ax.set_ylabel('Percentage of Comments (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
for bar in ax.patches:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{bar.get_height():.1f}%', ha='center', fontsize=10, fontweight='bold')

# Plot 2: Per-post breakdown
ax = axes[1]
pivot = all_df.groupby(['post','Sentiment Label']).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct[[c for c in ['Will Buy','Neutral / Undecided','Will Not Buy'] if c in pivot_pct.columns]
].plot(kind='bar', ax=ax,
       color=['steelblue','lightgray','tomato'],
       edgecolor='white', width=0.7)
ax.set_title('Purchase Intent by Post\n(V2 Improved)', fontweight='bold')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Sentiment', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

# Plot 3: V1 vs V2 comparison
ax = axes[2]
comparison_clean = comparison.dropna()
comparison_clean.plot(kind='bar', ax=ax,
                      color=['#aec6cf','#2196f3'],
                      edgecolor='white', width=0.7)
ax.set_title('V1 vs V2 Approach\nLabel Distribution', fontweight='bold')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.legend()

plt.suptitle('Valorant Skin Bundle — Reddit Sentiment Analysis',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('valorant_sentiment_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compound score distribution
fig, ax = plt.subplots(figsize=(10, 5))
for post, color in [('Bundle 1','steelblue'),('Bundle 2','darkorange'),('Bundle 3','green')]:
    subset = all_df[all_df['post']==post]['Compound Score']
    ax.hist(subset, bins=30, alpha=0.5, color=color, label=post, density=True)
ax.axvline( 0.05, color='black', linestyle='--', lw=1.2, label='Buy threshold (+0.05)')
ax.axvline(-0.05, color='red',   linestyle='--', lw=1.2, label='No-Buy threshold (-0.05)')
ax.set_xlabel('VADER Compound Score')
ax.set_ylabel('Density')
ax.set_title('Distribution of Sentiment Scores by Bundle Post', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('valorant_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Findings

### Results Across All Three Bundles (V2 Improved)

| Bundle Post | Will Buy | Neutral / Undecided | Will Not Buy |
|-------------|----------|---------------------|--------------|
| Bundle 1 (1b4p0vr) | ~38% | ~41% | ~21% |
| Bundle 2 (xibjap)  | ~41% | ~41% | ~18% |
| Bundle 3 (i2ykah)  | 35.2% | 32.7% | 32.1% |

### What Changed Between V1 and V2

| Issue | V1 Baseline | V2 Improved |
|-------|------------|-------------|
| Text cleaning | Removes basic emoticons only | Also removes URLs, Reddit quotes, broader emoji range |
| Negation handling | `"not good"` → counts as positive | VADER handles negations internally |
| Intensity | `"very bad"` = `"bad"` | VADER boosts score for intensifiers |
| Neutral zone | Score == 0 exactly (almost never) | ±0.05 VADER recommended threshold |
| Label | "Wants Changes" | "Neutral / Undecided" (more accurate) |

### Business Insight

Across all three bundles, roughly one-third of the community expresses positive purchase
intent, one-third is neutral, and one-third will not buy. Bundle 3 showed the most
evenly split distribution — a warning signal that the design may be dividing the community.
Bundles 1 and 2 had higher positive intent with fewer negative responses, suggesting
better community reception.

This pipeline can be run immediately after a bundle is announced to give Riot Games (or any
game studio) an early read on expected purchase conversion before the bundle goes live.

---

### Limitations

- VADER is trained on general English — gaming-specific slang (`W`, `L`, `fire`, `clean`)
  may be misclassified
- Reddit commenters are not representative of all players
- Purchase intent from a comment ≠ actual purchase behaviour
- Only top-level comments collected (`replace_more(limit=0)` excludes nested replies)